In [1]:
import cv2
import numpy as np
import matplotlib.pyplot as plt
import os
from sklearn.metrics.pairwise import cosine_similarity
from scipy.spatial.distance import euclidean

In [3]:
# 2: Load Dataset (Wang Corel-1K)
# Folder structure: dataset/0/, dataset/1/, ..., dataset/9/
# 0=Africans  1=Beach  2=Buildings  3=Buses  4=Dinosaurs
# 5=Elephants  6=Flowers  7=Horses  8=Mountains  9=Food

dataset_path = " "   # <-- UPDATE THIS PATH

images, labels, paths = [], [], []

for category in range(10):
    folder = os.path.join(dataset_path, str(category))
    for img_file in os.listdir(folder):
        img_path = os.path.join(folder, img_file)
        img = cv2.imread(img_path)
        if img is None:
            continue
        img = cv2.resize(img, (128, 128))
        images.append(img)
        labels.append(category)
        paths.append(img_path)

images = np.array(images)
labels = np.array(labels)

print(f"Loaded {len(images)} images from Wang Corel-1K dataset")
print(f"   Categories found: {np.unique(labels)}")

FileNotFoundError: [Errno 2] No such file or directory: ' /0'

In [4]:
#3: Preprocessing with CLAHE

def preprocess_image(img):
    """
    Applies CLAHE on the L channel (LAB color space)
    to normalize contrast before feature extraction.
    """
    lab = cv2.cvtColor(img, cv2.COLOR_BGR2LAB)
    l_channel, a_channel, b_channel = cv2.split(lab)

    clahe = cv2.createCLAHE(clipLimit=2.0, tileGridSize=(8, 8))
    cl = clahe.apply(l_channel)

    merged_lab = cv2.merge((cl, a_channel, b_channel))
    enhanced_img = cv2.cvtColor(merged_lab, cv2.COLOR_LAB2BGR)

    return enhanced_img

# Quick test
sample = preprocess_image(images[0])
print(f"CLAHE preprocessing OK. Output shape: {sample.shape}")

IndexError: list index out of range

In [5]:

#  4: Build Feature Index for All Dataset Images

print("Building feature index... please wait ⏳")

feature_index = []
for i, img in enumerate(images):
    feat = get_fused_features(img)
    feature_index.append(feat)
    if (i + 1) % 100 == 0:
        print(f"  Processed {i+1}/{len(images)} images...")

feature_index = np.array(feature_index)

# Save so you don't recompute every time
np.save('feature_index.npy', feature_index)
np.save('labels.npy', labels)

print(f"\n Feature index shape: {feature_index.shape}")
print(f" Saved to feature_index.npy and labels.npy")

Building feature index... please wait ⏳

 Feature index shape: (0,)
 Saved to feature_index.npy and labels.npy


In [ ]:

#  4: Color Histogram Feature Extraction

def extract_color_features(img):
    """
    Extracts a 96-dimensional HSV color histogram.
    32 bins x 3 channels = 96 features.
    """
    hsv = cv2.cvtColor(img, cv2.COLOR_BGR2HSV)
    hist_features = []

    for ch in range(3):
        h = cv2.calcHist([hsv], [ch], None, [32], [0, 256])
        cv2.normalize(h, h)
        hist_features.extend(h.flatten())

    return np.array(hist_features)  # shape: (96,)

# Quick test
color_vec = extract_color_features(images[0])
print(f" Color feature vector size: {color_vec.shape}")